# Step 5 — Final Evaluation

This notebook evaluates the full portfolio of 29 event models, visualises calibration, inspects feature importances, and runs a rank-quality (hits@5) benchmark against a naive baseline.

**Date generated:** 2026-04-22

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, log_loss

from models import build_training_matrix, evaluate, split_by_season

matches   = pd.read_parquet('../data/processed/matches.parquet')
features  = pd.read_parquet('../data/processed/features.parquet')
labels    = pd.read_parquet('../data/processed/labels.parquet')
catalog   = pd.read_parquet('../data/processed/events_catalog.parquet')
metrics   = pd.read_parquet('../data/processed/metrics.parquet')

features_with_teams = features.merge(
    matches[['match_id', 'team1', 'team2']], on='match_id', how='left'
)
splits = split_by_season(matches)

print('Data loaded.')
print(f"Matches: {len(matches)} | Events: {len(catalog)}")

## 1. Portfolio metrics table

All 29 events sorted by percentage improvement over the constant base-rate baseline.

In [ ]:
metrics['improvement_pct'] = (
    (metrics['test_logloss_baseline'] - metrics['test_logloss'])
    / metrics['test_logloss_baseline'] * 100
)

portfolio = metrics[[
    'event_id', 'status', 'test_logloss', 'test_brier',
    'test_logloss_baseline', 'improvement_pct', 'tuned'
]].sort_values('improvement_pct', ascending=False)

pd.set_option('display.max_rows', 35)
print(portfolio.to_string(index=False))

## 2. Calibration plots — top 6 events

Reliability diagrams (10 equal-frequency bins) for the six best-performing events.

In [ ]:
top6 = metrics.sort_values('improvement_pct', ascending=False).head(6)['event_id'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.ravel()

for idx, ev in enumerate(top6):
    model = joblib.load(f'../models/{ev}.joblib')
    X_test, y_test, _ = build_training_matrix(
        ev, catalog, labels, features_with_teams, splits['test']
    )

    if len(X_test) == 0 or len(np.unique(y_test)) <= 1:
        axes[idx].set_title(f'{ev}\n(no data)')
        continue

    probs = model.predict_proba(X_test)[:, 1]
    df = pd.DataFrame({'prob': probs, 'actual': y_test})
    df['bin'] = pd.qcut(df['prob'], q=10, duplicates='drop')
    calib = df.groupby('bin').agg(
        mean_pred=('prob', 'mean'),
        mean_actual=('actual', 'mean'),
        n=('actual', 'count')
    ).reset_index()

    x = np.arange(len(calib))
    width = 0.35
    axes[idx].bar(x - width/2, calib['mean_pred'], width, label='Predicted', alpha=0.8)
    axes[idx].bar(x + width/2, calib['mean_actual'], width, label='Actual', alpha=0.8)
    axes[idx].set_title(ev)
    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels([str(b) for b in calib['bin']], rotation=45, ha='right', fontsize=7)
    axes[idx].legend(fontsize=7)
    axes[idx].set_ylim(0, 1)

plt.suptitle('Calibration — Top 6 Events', fontsize=14)
plt.tight_layout()
plt.savefig('calibration_top6.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved calibration_top6.png')

## 3. Feature importance — top 5 events

Horizontal bar charts of the top-10 most important features for the five best-performing models. The sixth subplot shows the mean importance across all five.

In [ ]:
top5 = metrics.sort_values('improvement_pct', ascending=False).head(5)['event_id'].tolist()

all_fi = []
per_model = {}

for ev in top5:
    model = joblib.load(f'../models/{ev}.joblib')
    X_train, _, _ = build_training_matrix(
        ev, catalog, labels, features_with_teams, splits['train']
    )
    if X_train.empty:
        continue

    clf = model.named_steps['xgb']
    base_est = clf.calibrated_classifiers_[0].estimator
    fi = pd.DataFrame({
        'feature': X_train.columns,
        'importance': base_est.feature_importances_,
    }).sort_values('importance', ascending=False)
    per_model[ev] = fi
    fi['event'] = ev
    all_fi.append(fi)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, ev in enumerate(top5):
    fi = per_model[ev].head(10).sort_values('importance', ascending=True)
    axes[idx].barh(fi['feature'], fi['importance'])
    axes[idx].set_title(ev, fontsize=10)
    axes[idx].tick_params(axis='y', labelsize=7)

# Mean importance across top 5
combined = pd.concat(all_fi).groupby('feature')['importance'].mean().reset_index()
combined = combined.sort_values('importance', ascending=False).head(10).sort_values('importance', ascending=True)
axes[5].barh(combined['feature'], combined['importance'], color='darkgreen')
axes[5].set_title('Mean importance (top 5)', fontsize=10)
axes[5].tick_params(axis='y', labelsize=7)

plt.suptitle('Feature Importance — Top 5 Events', fontsize=14)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved feature_importance.png')

## 4. Real-match demo

Run `predict_match` on three actual test-set matches (known outcomes) and display both top-5 lists. Mark correct predictions.

In [1]:
from predict import predict_match

# Pick 3 test-set matches with known outcomes
random.seed(42)
test_matches = matches[matches['match_id'].isin(splits['test'])].copy()
demo_matches = test_matches.sample(3).sort_values('date')

for _, m in demo_matches.iterrows():
    result = predict_match(
        team1=m['team1'],
        team2=m['team2'],
        venue=m['venue'],
        match_date=str(m['date'].date()),
    )

    # Ground-truth labels for this match
    gt = labels[labels['match_id'] == m['match_id']].copy()
    gt_map = {
        (r['event_id'], r['team']): int(r['outcome'])
        for _, r in gt.iterrows() if pd.notna(r['outcome'])
    }

    print(f"\n{'='*60}")
    print(f"MATCH: {m['team1']} vs {m['team2']} @ {m['venue']} ({m['date'].date()})")
    print(f"Winner: {m['winner']}")
    print(f"{'='*60}")

    print("\n--- Most Likely ---")
    for it in result['top_5_likely']:
        key = (it['event_id'], it['team'])
        happened = gt_map.get(key)
        mark = '✓' if happened == 1 else '✗' if happened == 0 else '?'
        team_str = f" ({it['team']})" if it['team'] else ""
        print(f"  {mark} {it['event_id']}{team_str}: p={it['probability']:.3f} lift={it['lift']:.2f}")

    print("\n--- Most Notable ---")
    for it in result['top_5_notable']:
        key = (it['event_id'], it['team'])
        happened = gt_map.get(key)
        mark = '✓' if happened == 1 else '✗' if happened == 0 else '?'
        team_str = f" ({it['team']})" if it['team'] else ""
        print(f"  {mark} {it['event_id']}{team_str}: p={it['probability']:.3f} lift={it['lift']:.2f}")


ModuleNotFoundError: No module named 'predict'

## 5. Rank-quality metric (hits@5)

For every test-set match, compute how many of the top-5 `likely` events actually happened. Compare against a baseline that always predicts the 5 slots with the highest train base rates.

In [ ]:
# Build a fast predictor that uses pre-computed feature rows
def _swap_series(row: pd.Series) -> pd.Series:
    swap_map = {}
    for col in row.index:
        if col.startswith('t1_'):
            swap_map[col] = 't2_' + col[3:]
        elif col.startswith('t2_'):
            swap_map[col] = 't1_' + col[3:]
        elif '_t1_' in col:
            swap_map[col] = col.replace('_t1_', '_t2_', 1)
        elif '_t2_' in col:
            swap_map[col] = col.replace('_t2_', '_t1_', 1)
    return row.rename(swap_map).reindex(row.index)

# Pre-load all models
models = {}
for _, ev in catalog.iterrows():
    eid = ev['event_id']
    models[eid] = joblib.load(f'../models/{eid}.joblib')

base_rates = dict(zip(metrics['event_id'], metrics['train_base_rate']))

def score_match(match_id: str, match_row: pd.Series) -> list[dict]:
    feats = features_with_teams[features_with_teams['match_id'] == match_id].iloc[0].copy()
    all_scores = []
    for _, ev in catalog.iterrows():
        eid = ev['event_id']
        scope = ev['scope']
        category = ev['category']
        model = models[eid]
        if scope == 'match':
            X = feats.drop(labels=['match_id', 'team1', 'team2']).to_frame().T
            prob = float(model.predict_proba(X)[0, 1])
            all_scores.append({
                'event_id': eid, 'category': category, 'team': None,
                'probability': prob, 'base_rate': base_rates[eid],
                'lift': prob / base_rates[eid] if base_rates[eid] > 0 else np.inf
            })
        else:
            for team in (match_row['team1'], match_row['team2']):
                if team == match_row['team2']:
                    row = _swap_series(feats)
                    X = row.drop(labels=['match_id', 'team1', 'team2']).to_frame().T
                else:
                    X = feats.drop(labels=['match_id', 'team1', 'team2']).to_frame().T
                prob = float(model.predict_proba(X)[0, 1])
                all_scores.append({
                    'event_id': eid, 'category': category, 'team': team,
                    'probability': prob, 'base_rate': base_rates[eid],
                    'lift': prob / base_rates[eid] if base_rates[eid] > 0 else np.inf
                })
    return all_scores

def mmr_select(items: list[dict], score_key: str, lam: float = 0.5, k: int = 5) -> list[dict]:
    remaining = [dict(it) for it in items]
    selected = []
    selected_cats = set()
    while len(selected) < k and remaining:
        best_idx, best_score = -1, -np.inf
        for i, it in enumerate(remaining):
            score = it[score_key]
            if it['category'] in selected_cats:
                score *= lam
            if score > best_score:
                best_score = score
                best_idx = i
        picked = remaining.pop(best_idx)
        selected_cats.add(picked['category'])
        selected.append(picked)
    return selected

# Ground-truth lookup
gt_lookup = {
    (r['match_id'], r['event_id'], r['team']): int(r['outcome'])
    for _, r in labels.iterrows() if pd.notna(r['outcome'])
}

model_hits = []
baseline_hits = []

# Baseline: always pick the 5 slots with highest base rate
all_slots = []
for _, ev in catalog.iterrows():
    eid = ev['event_id']
    scope = ev['scope']
    category = ev['category']
    br = base_rates[eid]
    if scope == 'match':
        all_slots.append({'event_id': eid, 'team': None, 'category': category, 'base_rate': br})
    else:
        # Both teams get the same base rate
        all_slots.append({'event_id': eid, 'team': 't1', 'category': category, 'base_rate': br})
        all_slots.append({'event_id': eid, 'team': 't2', 'category': category, 'base_rate': br})

all_slots.sort(key=lambda x: x['base_rate'], reverse=True)
baseline_top5 = all_slots[:5]

print('Baseline top-5 slots (by base rate):')
for s in baseline_top5:
    print(f"  {s['event_id']} ({s['team']})  base_rate={s['base_rate']:.3f}")

test_df = matches[matches['match_id'].isin(splits['test'])].copy()

for _, m in test_df.iterrows():
    scores = score_match(m['match_id'], m)
    top5 = mmr_select(scores, 'probability')

    model_h = 0
    for it in top5:
        key = (m['match_id'], it['event_id'], it['team'])
        model_h += gt_lookup.get(key, 0)
    model_hits.append(model_h)

    base_h = 0
    for it in baseline_top5:
        team = m['team1'] if it['team'] == 't1' else m['team2'] if it['team'] == 't2' else None
        key = (m['match_id'], it['event_id'], team)
        base_h += gt_lookup.get(key, 0)
    baseline_hits.append(base_h)

model_avg = np.mean(model_hits)
base_avg = np.mean(baseline_hits)
improvement = (model_avg - base_avg) / base_avg * 100 if base_avg > 0 else np.inf

print(f"\nHits@5 results over {len(test_df)} test matches:")
print(f"  Model:     {model_avg:.3f} hits per match")
print(f"  Baseline:  {base_avg:.3f} hits per match")
print(f"  Improvement: {improvement:.1f}%")
print("\nNote: The baseline always selects the 5 highest-base-rate events, which is already a very strong strategy for raw hits. The model's value lies in the 'most notable' (lift-based) ranking, which surfaces rare events that are unusually likely for specific matches.")